# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, a clutch term, and consistency. A career rate is the decay-weighted mean of the rows that player has. A season with no rows is left out of both the total and the weight. The weights are 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or an other-type extra. Super overs (innings 3 and 4) are kept; every super-over ball is a death ball. An other-type extra is re-bowled like a no-ball. Every extra carries its run amount, so a team total includes wides, no-balls, byes, and leg-byes. Every run off a wide is charged to the bowler, and so is the one run of a no-ball. Byes run off a no-ball are not. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied. The second super over chases the first super over's total plus one.

Pressure has three parts: the situation (occasion times competition times the ball), a shock from the last six balls, and a break from the previous eighteen. Each part is divided by its own spread and weighted 0.60, 0.25 and 0.15, so the match situation leads and a recent wicket or boundary raises pressure without deciding the band on its own. The current ball is not included. The shock starts with dots and wickets for a batter, and fours and sixes for a bowler, then adds the wickets above xW and the charged runs above xR on those same six balls. Bowling bands are cut inside pace and inside spin, so the new ball does not put more pace bowlers into the high band. A recognized batter is anyone at positions 1 to 7. Zero, medium, and high are the bottom half, the next 30 percent, and the top fifth of that pressure.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal", len(legal), "of which super overs", int(legal["is_super_over"].sum()))
print("five-season weight total", round(ratings.W_SUM, 4))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))

deliveries 200999 legal 192445 of which super overs 33
five-season weight total 4.0951


,balls,share
band,,
zero,95199,49.5
medium,58746,30.5
high,38500,20.0


## Elo

Team strength updates after each match. Batting strength and bowling strength update after each legal ball, on three ladders: IPL, T20I, and lower T20. Emerging Teams and Rising Stars are the lower ladder. A rating learned there does not carry into a T20I, and a ball there counts as half a ball in the career rates and the ranking floor. A player with few balls is held near 1500, and that hold loosens as the career lengthens. Venue strength uses only balls already bowled at that ground. Expected runs and the expected wicket chance are also taken from balls already bowled, split by phase, pace or spin, and ladder, and pulled toward the coarser rate with 30 balls. Before any ball, the starting assumption is 1.0 run and a 0.05 wicket chance. A batter is priced on bat runs. A bowler is priced on charged runs, which are 0 on a bye, leg-bye, or penalty, and the bowling rating uses that same charged total.

In [2]:
charged_gap = legal["bat_runs"] - legal["charged_runs"]
print("balls where bat runs differ from charged runs", int(charged_gap.ne(0).sum()))
print("sum of bat runs minus charged runs", round(float(charged_gap.sum()), 3))
legal = ratings.fit_elo(legal)
summary = legal.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))

balls where bat runs differ from charged runs 1
sum of bat runs minus charged runs 1.0


k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 1043.2
batter_elo_max: 2500.4
batter_elo_sd: 124.8
bowler_elo_min: 1062.5
bowler_elo_max: 2082.6
bowler_elo_sd: 113.2
team_elo_min: 1361.5
team_elo_max: 1647.1
grounds: 97
step_sizes: as written


## Batter scores

Baseline is strike rate and average against the phases that batter actually plays. The three bands are the strength-adjusted ball value inside zero, medium, and high pressure. Peers compare an innings with teammates and the opposition, and the opposition gap is capped at 3, the same cap as the teammate gap. Clutch uses a high-band spell against the player's own rate. A spell below that rate stays negative. A spell above it is scaled by how far the player also beat their teammates, and a negative teammate gap does not add a bonus. Consistency prefers a narrower spread of match values.

Each term is a z-score that is half inside the batting role and half against every batter. The role half judges a lower-order hitter partly against others doing the same job; the pooled half stops a weak role group from lifting him on its own. The split is a modelling choice, not a finding. The reference is players with at least 30 decay-weighted legal balls.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 141


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Tim David,middle order,1.577,2.284,1.563,1.708,0.978,1.935,2.920,-0.699,2142,177.025,34.548,998.270,338.486
1,2.0,Phil Salt,top order,1.283,1.960,0.742,0.929,1.271,2.978,0.968,-1.204,2764,169.779,36.853,1357.107,61.552
2,3.0,Rinku Singh,middle order,1.164,1.700,0.382,-0.218,2.139,3.000,0.693,-1.289,1945,152.669,38.900,1045.702,306.270
3,4.0,Donovan Ferreira,middle order,1.162,0.956,0.053,1.707,1.995,3.000,1.015,-2.414,561,164.516,24.391,316.689,127.002
4,5.0,Jos Buttler,top order,1.065,1.899,0.544,-0.088,1.240,2.362,0.798,-0.382,4384,155.461,41.752,2303.778,221.244
5,6.0,Reeza Hendricks,top order,0.981,1.093,-0.117,1.122,1.474,2.556,-0.148,-0.579,1220,146.635,35.882,630.953,83.119
6,7.0,Glenn Phillips,middle order,0.921,1.298,1.070,0.241,0.102,3.000,0.514,-0.030,1613,147.982,35.065,885.551,128.185
7,8.0,David Miller,middle order,0.913,1.589,0.795,-0.056,1.100,2.482,-0.336,-0.348,1898,149.097,38.735,975.103,308.662
8,9.0,George Linde,lower order,0.910,0.750,0.161,0.410,1.969,1.111,1.754,-0.530,227,162.143,18.917,130.740,62.940
9,10.0,Sai Sudharsan,top order,0.848,2.021,0.157,0.301,0.772,0.745,-0.374,1.541,2496,149.104,48.941,1448.056,142.160


roles in the batter top 10
role
middle order    5
top order       4
lower order     1


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's pace or spin and the phases they bowl. A lower rate is a higher score. The high band, clutch, and peers use the same direction.

In [4]:
bowlers = ratings.score_bowlers(legal, work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 143


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Mitchell Santner,Spin,1.055,0.549,0.091,0.320,1.101,3.000,1.912,0.271,87.0,7.444,19.966,1425.581,268.639
1,2.0,Shaheen Afridi,Pace,0.837,1.255,0.611,0.164,1.313,0.991,0.694,0.195,65.0,8.012,15.231,840.402,202.158
2,3.0,Bhuvneshwar Kumar,Pace,0.780,0.647,0.314,0.789,0.454,1.554,1.734,0.033,115.0,8.291,19.313,1771.245,612.871
3,4.0,Abrar Ahmed,Spin,0.769,1.391,0.951,-0.378,1.015,0.261,1.879,0.222,46.0,6.683,16.609,689.250,137.350
4,5.0,Jasprit Bumrah,Pace,0.759,1.285,0.655,1.606,-0.149,-0.513,2.940,0.084,85.0,7.268,20.094,1454.183,320.063
5,6.0,Mohammad Nawaz,Spin,0.758,1.284,1.033,-0.716,1.527,0.771,1.129,-0.287,66.0,7.124,15.212,812.210,149.387
6,7.0,Mark Adair,Pace,0.737,0.794,0.137,0.774,0.413,2.134,-0.487,0.941,56.0,8.720,15.089,645.510,102.494
7,8.0,Sunil Narine,Spin,0.729,0.565,1.243,0.109,0.621,-0.560,3.000,1.353,66.0,6.865,23.227,1249.702,300.720
8,9.0,Mujeeb Ur Rahman,Spin,0.712,1.253,0.072,0.298,0.571,0.455,1.691,0.583,48.0,7.032,17.083,675.993,175.069
9,10.0,Nathan Ellis,Pace,0.705,1.041,0.569,0.736,0.497,0.129,1.187,0.917,58.0,8.253,15.793,735.308,87.005


roles in the bowler top 10
role
Spin    5
Pace    5


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 60 of those in the high band. Anyone under the line stays in the full table with no rank. The two lists are saved under `ratings/`.

In [5]:
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: career rate is the decay-weighted mean of rows played; the five weights sum to", round(ratings.W_SUM, 4))
print("decision: elo step sizes", summary["step_sizes"])

wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: career rate is the decay-weighted mean of rows played; the five weights sum to 4.0951
decision: elo step sizes as written


## Deck exports

These files are the evidence behind each stage: cleaning counts, pressure distributions, Elo bounds, calibration, component weights, and the rankings. They are written from the frames above.

In [6]:
notes = ratings.write_deck_exports(balls, work, legal, batters, bowlers)
print("wrote exports/")
for item, value in notes.items():
    print(item + ":", value)

wrote exports/
sample_match_id: 40767
sample_innings: 2
sample_date: 2025-06-03
sample_stage: Final
sample_competition: IPL 2025
sample_home: Royal Challengers Bengaluru
sample_away: Punjab Kings
trajectory_player: Jos Buttler
trajectory_up: 31250
trajectory_down: 20206
trajectory_up_date: 2024-04-16
trajectory_down_date: 2023-04-19
trajectory_up_runs: 107.0
trajectory_down_runs: 36.0
